# HMM classification model

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from hmmlearn.hmm import GaussianHMM
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler


CSV_PATH = Path("FMIX.csv")
HOLDOUT_FRACTION = 0.20
N_SPLITS = 4
N_AHEAD = 20
RANDOM_STATE = 42

# n_ahead is a forecasting objective, not a model hyperparameter. Keep it fixed
# while comparing candidates so all models solve the same task.
EXPANDED_PARAM_GRID = {
    "look_forward": [40, 60, 80, 100],
    "prediction_length": [20, 40, 60, 80],
    "n_components": [2, 3, 4, 5],
    "covariance_type": ["diag", "tied"],
}

print("Количество конфигураций:",
      len(list(ParameterGrid(EXPANDED_PARAM_GRID))))

FEATURE_COLUMNS = [
    "log_return_1",
    "log_return_3",
    "log_return_5",
    "high_low_range",
    "candle_body",
    "upper_wick",
    "lower_wick",
    "log_volume_change",
    "rolling_volatility_10",
    "rolling_volatility_20",
    "volume_zscore_20",
    "hour_sin",
    "hour_cos",
]


Количество конфигураций: 128


In [2]:
def load_market_data(csv_path):
    """Load OHLCV data and use a unique, sorted DatetimeIndex."""
    df = pd.read_csv(csv_path)
    required = {
        "<DATE>", "<TIME>", "<OPEN>", "<HIGH>",
        "<LOW>", "<CLOSE>", "<VOL>",
    }
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f"Missing input columns: {sorted(missing)}")

    date_text = df["<DATE>"].astype(str).str.replace(r"\.0$", "", regex=True)
    time_text = (
        df["<TIME>"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(6)
    )
    timestamp = pd.to_datetime(
        date_text + " " + time_text,
        format="%Y%m%d %H%M%S",
        errors="raise",
    )

    market = df.rename(
        columns={
            "<OPEN>": "X.OPEN",
            "<HIGH>": "X.HIGH",
            "<LOW>": "X.LOW",
            "<CLOSE>": "X.CLOSE",
            "<VOL>": "X.VOL",
        }
    )[["X.OPEN", "X.HIGH", "X.LOW", "X.CLOSE", "X.VOL"]].copy()
    market.index = pd.DatetimeIndex(timestamp, name="timestamp")
    market = market.sort_index()

    if market.index.has_duplicates:
        duplicate_count = int(market.index.duplicated(keep="last").sum())
        warnings.warn(f"Dropping {duplicate_count} duplicate timestamps")
        market = market[~market.index.duplicated(keep="last")]

    market = market.apply(pd.to_numeric, errors="coerce")
    if market.isna().any().any():
        bad_rows = int(market.isna().any(axis=1).sum())
        raise ValueError(f"OHLCV contains non-numeric or missing values in {bad_rows} rows")
    if (market[["X.OPEN", "X.HIGH", "X.LOW", "X.CLOSE"]] <= 0).any().any():
        raise ValueError("OHLC prices must be positive for logarithmic features")
    if (market["X.VOL"] <= 0).any():
        warnings.warn("Non-positive volume rows will be removed by feature preparation")

    return market


def build_features(market, session_start=9, session_end=23):
    """Причинные стационарные признаки с единым DatetimeIndex."""
    features = pd.DataFrame(index=market.index)

    log_open = np.log(market["X.OPEN"])
    log_high = np.log(market["X.HIGH"])
    log_low = np.log(market["X.LOW"])
    log_close = np.log(market["X.CLOSE"])

    positive_volume = market["X.VOL"].where(market["X.VOL"] > 0)
    log_volume = np.log(positive_volume)

    # Доходности за несколько коротких горизонтов.
    features["log_return_1"] = log_close.diff(1)
    features["log_return_3"] = log_close.diff(3)
    features["log_return_5"] = log_close.diff(5)

    # Геометрия свечи в логарифмическом масштабе.
    features["high_low_range"] = log_high - log_low
    features["candle_body"] = log_close - log_open

    max_open_close = pd.concat([log_open, log_close], axis=1).max(axis=1)
    min_open_close = pd.concat([log_open, log_close], axis=1).min(axis=1)

    features["upper_wick"] = log_high - max_open_close
    features["lower_wick"] = min_open_close - log_low

    # Изменение и относительная аномальность объёма.
    features["log_volume_change"] = log_volume.diff()

    volume_mean_20 = log_volume.rolling(20, min_periods=20).mean()
    volume_std_20 = log_volume.rolling(20, min_periods=20).std(ddof=0)
    volume_std_20 = volume_std_20.replace(0, np.nan)

    features["volume_zscore_20"] = (
        log_volume - volume_mean_20
    ) / volume_std_20

    # Реализованная волатильность только по прошлым и текущим данным.
    features["rolling_volatility_10"] = (
        features["log_return_1"]
        .rolling(10, min_periods=10)
        .std(ddof=0)
    )
    features["rolling_volatility_20"] = (
        features["log_return_1"]
        .rolling(20, min_periods=20)
        .std(ddof=0)
    )

    # Циклическое представление времени внутри торговой сессии.
    hour = pd.Series(features.index.hour, index=features.index)
    session_length = session_end - session_start + 1
    session_position = hour - session_start

    features["hour_sin"] = np.sin(
        2 * np.pi * session_position / session_length
    )
    features["hour_cos"] = np.cos(
        2 * np.pi * session_position / session_length
    )

    features = features.loc[hour.between(session_start, session_end)]

    return (
        features
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=FEATURE_COLUMNS)
    )


def make_extrema_labels(market, look_forward):
    """Create retrospective long/short regimes from forward local extrema.

    A local minimum starts a long regime and a local maximum starts a short
    regime. The last look_forward rows remain unknown because their complete
    forward window is unavailable.
    """
    if look_forward < 1:
        raise ValueError("look_forward must be positive")
    if len(market) <= look_forward:
        raise ValueError("Not enough rows to build extrema labels")

    close = market["X.CLOSE"]
    events = pd.Series(pd.NA, index=market.index, dtype="object")
    last_start = len(close) - look_forward

    for i in range(last_start):
        window = close.iloc[i : i + look_forward + 1]
        current = close.iloc[i]
        window_min = window.min()
        window_max = window.max()
        if window_min == window_max:
            continue
        if current == window_min:
            events.iloc[i] = "long"
        elif current == window_max:
            events.iloc[i] = "short"

    labels = events.ffill().rename("label")
    labels.iloc[-look_forward:] = pd.NA
    return labels


def assemble_dataset(market, look_forward):
    """Join features, close prices and labels by timestamp, never by position."""
    features = build_features(market)
    labels = make_extrema_labels(market, look_forward)
    dataset = features.join(market[["X.CLOSE"]], how="inner").join(labels, how="left")
    dataset = dataset.dropna(subset=FEATURE_COLUMNS + ["X.CLOSE"])
    if not dataset.index.is_monotonic_increasing or dataset.index.has_duplicates:
        raise ValueError("Prepared dataset must have a sorted unique DatetimeIndex")
    return dataset


In [3]:
class HMMDirectionClassifier:
    """Two class-conditional HMMs with one scaler fitted on training data."""

    def __init__(
        self,
        n_components=3,
        covariance_type="diag",
        n_iter=200,
        random_state=42,
        min_sequence_length=3,
    ):
        self.n_components = int(n_components)
        self.covariance_type = covariance_type
        self.n_iter = int(n_iter)
        self.random_state = int(random_state)
        self.min_sequence_length = int(min_sequence_length)
        self.scaler = StandardScaler()
        self.models = {}

    def _extract_sequences(self, frame, direction):
        labeled = frame.dropna(subset=["label"]).copy()
        if labeled.empty:
            raise ValueError("Training fold has no labeled rows")

        run_id = labeled["label"].ne(labeled["label"].shift()).cumsum()
        sequences = []
        minimum_length = max(self.min_sequence_length, self.n_components + 1)
        for _, run in labeled.groupby(run_id, sort=False):
            if run["label"].iloc[0] != direction:
                continue
            values = run[FEATURE_COLUMNS].to_numpy(dtype=float)
            if len(values) >= minimum_length:
                sequences.append(values)

        if not sequences:
            raise ValueError(
                f"No {direction!r} sequences with at least {minimum_length} rows"
            )
        return sequences

    def _minimum_sample_count(self):
        n_features = len(FEATURE_COLUMNS)
        if self.covariance_type == "full":
            per_state = n_features + n_features * (n_features + 1) // 2 + 1
        else:
            per_state = 2 * n_features + 1
        return self.n_components * per_state

    def fit(self, train_frame):
        labeled = train_frame.dropna(subset=["label"])
        if labeled["label"].nunique() < 2:
            raise ValueError("Training fold must contain both long and short labels")
        if not np.isfinite(labeled[FEATURE_COLUMNS].to_numpy()).all():
            raise ValueError("Training features contain NaN or infinite values")

        self.scaler.fit(labeled[FEATURE_COLUMNS].to_numpy(dtype=float))
        minimum_samples = self._minimum_sample_count()
        self.models = {}

        for direction in ("long", "short"):
            raw_sequences = self._extract_sequences(labeled, direction)
            scaled_sequences = [self.scaler.transform(seq) for seq in raw_sequences]
            lengths = np.asarray([len(seq) for seq in scaled_sequences], dtype=int)
            observations = np.vstack(scaled_sequences)

            if len(observations) < minimum_samples:
                raise ValueError(
                    f"{direction!r} has {len(observations)} observations; "
                    f"at least {minimum_samples} are required for "
                    f"{self.n_components} states and {self.covariance_type!r} covariance"
                )
            if lengths.sum() != len(observations) or (lengths <= 0).any():
                raise ValueError(f"Invalid HMM sequence lengths for {direction!r}")

            model = GaussianHMM(
                n_components=self.n_components,
                covariance_type=self.covariance_type,
                n_iter=self.n_iter,
                min_covar=1e-5,
                random_state=self.random_state,
            )
            model.fit(observations, lengths=lengths)
            if not model.monitor_.converged:
                raise RuntimeError(f"The {direction!r} HMM did not converge")
            self.models[direction] = model
        return self

    def predict_window(self, feature_window):
        if set(self.models) != {"long", "short"}:
            raise RuntimeError("Fit the classifier before prediction")
        observations = self.scaler.transform(
            feature_window[FEATURE_COLUMNS].to_numpy(dtype=float)
        )
        if len(observations) < 2 or not np.isfinite(observations).all():
            raise ValueError("Prediction window is empty, too short or non-finite")

        # Normalization makes the value interpretable if window lengths are
        # compared in diagnostics; it does not change the winner here.
        scores = {
            direction: model.score(observations) / len(observations)
            for direction, model in self.models.items()
        }
        prediction = max(scores, key=scores.get)
        return prediction, scores


In [4]:
def direction_from_delta(delta):
    if pd.isna(delta):
        return pd.NA
    if delta > 0:
        return "long"
    if delta < 0:
        return "short"
    return "flat"


def rolling_predictions(model, frame, prediction_length, n_ahead, start_time=None):
    """Predict at time t and evaluate close[t+1]-close[t] and close[t+h]-close[t]."""
    if prediction_length < 2:
        raise ValueError("prediction_length must be at least 2")
    if n_ahead < 1:
        raise ValueError("n_ahead must be positive")
    if len(frame) < prediction_length:
        return pd.DataFrame()

    rows = []
    close = frame["X.CLOSE"]
    for end_pos in range(prediction_length - 1, len(frame)):
        timestamp = frame.index[end_pos]
        if start_time is not None and timestamp < start_time:
            continue

        window = frame.iloc[end_pos - prediction_length + 1 : end_pos + 1]
        prediction, scores = model.predict_window(window)

        one_step_delta = (
            close.iloc[end_pos + 1] - close.iloc[end_pos]
            if end_pos + 1 < len(frame)
            else np.nan
        )
        horizon_delta = (
            close.iloc[end_pos + n_ahead] - close.iloc[end_pos]
            if end_pos + n_ahead < len(frame)
            else np.nan
        )
        rows.append(
            {
                "timestamp": timestamp,
                "prediction": prediction,
                "label": frame["label"].iloc[end_pos],
                "close_t": close.iloc[end_pos],
                "one_step_direction": direction_from_delta(one_step_delta),
                "future_direction": direction_from_delta(horizon_delta),
                "score_long": scores["long"],
                "score_short": scores["short"],
            }
        )

    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index("timestamp")


def binary_metrics(predictions, truth_column, prefix):
    if predictions.empty:
        return {
            f"{prefix}_count": 0,
            f"{prefix}_coverage": 0.0,
            f"{prefix}_accuracy": np.nan,
            f"{prefix}_balanced_accuracy": np.nan,
            f"{prefix}_f1_macro": np.nan,
        }

    valid = predictions[truth_column].isin(["long", "short"])
    evaluated = predictions.loc[valid]
    coverage = len(evaluated) / len(predictions)
    if evaluated.empty:
        return {
            f"{prefix}_count": 0,
            f"{prefix}_coverage": coverage,
            f"{prefix}_accuracy": np.nan,
            f"{prefix}_balanced_accuracy": np.nan,
            f"{prefix}_f1_macro": np.nan,
        }

    y_true = evaluated[truth_column]
    y_pred = evaluated["prediction"]
    return {
        f"{prefix}_count": len(evaluated),
        f"{prefix}_coverage": coverage,
        f"{prefix}_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        f"{prefix}_f1_macro": f1_score(
            y_true, y_pred, labels=["long", "short"], average="macro", zero_division=0
        ),
    }


def evaluate_predictions(predictions):
    metrics = {}
    metrics.update(binary_metrics(predictions, "label", "regime"))
    metrics.update(binary_metrics(predictions, "one_step_direction", "one_step"))
    metrics.update(binary_metrics(predictions, "future_direction", "future"))
    return metrics


In [5]:
def temporal_cv_search(
    development_market,
    param_grid,
    n_ahead,
    n_splits=4,
    random_state=42,
):
    """Select model parameters using development data only."""
    results = []
    for candidate_id, params in enumerate(ParameterGrid(param_grid), start=1):
        look_forward = int(params["look_forward"])
        prediction_length = int(params["prediction_length"])
        gap = max(look_forward, n_ahead)
        dataset = assemble_dataset(development_market, look_forward)

        splitter = TimeSeriesSplit(n_splits=n_splits, gap=gap)
        fold_metrics = []
        error_message = None

        try:
            for fold, (train_idx, test_idx) in enumerate(splitter.split(dataset), start=1):
                train_frame = dataset.iloc[train_idx]
                test_frame = dataset.iloc[test_idx]
                if train_frame.index.max() >= test_frame.index.min():
                    raise RuntimeError("Temporal split overlap detected")

                classifier = HMMDirectionClassifier(
                    n_components=params["n_components"],
                    covariance_type=params["covariance_type"],
                    random_state=random_state,
                ).fit(train_frame)
                predictions = rolling_predictions(
                    classifier,
                    test_frame,
                    prediction_length=prediction_length,
                    n_ahead=n_ahead,
                )
                metrics = evaluate_predictions(predictions)
                metrics["fold"] = fold
                fold_metrics.append(metrics)
        except Exception as exc:
            error_message = f"{type(exc).__name__}: {exc}"

        row = {
            "candidate_id": candidate_id,
            **params,
            "n_ahead": n_ahead,
            "gap": gap,
            "valid_folds": len(fold_metrics),
            "status": "ok" if len(fold_metrics) == n_splits else "failed",
            "error": error_message,
        }
        for metric_name in (
            "regime_accuracy",
            "regime_balanced_accuracy",
            "regime_f1_macro",
            "one_step_accuracy",
            "one_step_balanced_accuracy",
            "one_step_f1_macro",
            "future_accuracy",
            "future_balanced_accuracy",
            "future_f1_macro",
            "future_coverage",
        ):
            values = [fold_result[metric_name] for fold_result in fold_metrics]
            row[f"cv_{metric_name}"] = np.nanmean(values) if values else np.nan
        results.append(row)

        print(
            f"[{candidate_id:03d}] {row['status']}: {params}; "
            f"regime balanced accuracy="
            f"{row['cv_regime_balanced_accuracy']:.4f}"
            if row["status"] == "ok"
            else f"[{candidate_id:03d}] failed: {params}; {error_message}"
        )

    result_frame = pd.DataFrame(results)
    successful = result_frame[result_frame["status"] == "ok"]
    if successful.empty:
        errors = result_frame[["candidate_id", "error"]].to_dict("records")
        raise RuntimeError(f"Every CV candidate failed: {errors}")

    return result_frame.sort_values(
        [
            "cv_regime_balanced_accuracy",
            "cv_regime_f1_macro",
            "cv_future_balanced_accuracy",
        ],
        ascending=False,
        na_position="last",
    ).reset_index(drop=True)


## Parameter selection on the development period

The chronological holdout is removed before the grid search. Candidates are
ranked by mean balanced accuracy for the fixed `N_AHEAD` future direction.
`diag` covariance is the stable default; add `full` only after the corrected
pipeline works and enough observations are available for every state.


In [6]:
market = load_market_data(CSV_PATH)
split_position = int(len(market) * (1 - HOLDOUT_FRACTION))
development_market = market.iloc[:split_position].copy()
holdout_market = market.iloc[split_position:].copy()

print(f"All data:    {market.index.min()} -> {market.index.max()} ({len(market)} rows)")
print(
    f"Development: {development_market.index.min()} -> "
    f"{development_market.index.max()} ({len(development_market)} rows)"
)
print(
    f"Holdout:     {holdout_market.index.min()} -> "
    f"{holdout_market.index.max()} ({len(holdout_market)} rows)"
)

expanded_cv_results = temporal_cv_search(
    development_market=development_market,
    param_grid=EXPANDED_PARAM_GRID,
    n_ahead=N_AHEAD,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

expanded_cv_results.to_csv(
    "hmm_cv_results_expanded_features.csv",
    index=False,
)

columns_to_show = [
    "look_forward",
    "prediction_length",
    "n_components",
    "covariance_type",
    "valid_folds",
    "cv_regime_balanced_accuracy",
    "cv_future_balanced_accuracy",
    "cv_future_f1_macro",
]

display(expanded_cv_results[columns_to_show].head(15))


All data:    2023-01-03 09:00:00 -> 2025-04-11 23:00:00 (8537 rows)
Development: 2023-01-03 09:00:00 -> 2024-10-24 22:00:00 (6829 rows)
Holdout:     2024-10-24 23:00:00 -> 2025-04-11 23:00:00 (1708 rows)
[001] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 2, 'prediction_length': 20}; regime balanced accuracy=0.6162
[002] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 2, 'prediction_length': 40}; regime balanced accuracy=0.5773
[003] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 2, 'prediction_length': 60}; regime balanced accuracy=0.5458
[004] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 2, 'prediction_length': 80}; regime balanced accuracy=0.5303
[005] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 20}; regime balanced accuracy=0.6426
[006] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 40}; regime balanced ac

,look_forward,prediction_length,n_components,covariance_type,valid_folds,cv_regime_balanced_accuracy,cv_future_balanced_accuracy,cv_future_f1_macro
0,40,20,3,diag,4,0.642615,0.479669,0.475475
1,60,20,3,diag,4,0.638259,0.473416,0.471345
2,100,40,4,diag,4,0.633841,0.487487,0.472592
3,40,20,5,diag,4,0.630944,0.478891,0.476645
4,60,40,3,diag,4,0.625729,0.493104,0.483940
5,100,20,4,diag,4,0.622703,0.484016,0.472295
6,100,40,3,diag,4,0.621435,0.490125,0.482614
7,40,20,4,diag,4,0.617928,0.482086,0.475238
8,40,20,2,diag,4,0.616243,0.469928,0.457011
9,40,40,3,diag,4,0.614453,0.498276,0.491672


## One final holdout evaluation

The best CV parameters are fitted once on all development data. The holdout
receives preceding context only to construct the first prediction window;
holdout observations never enter model fitting or parameter selection.


In [7]:
best_row = expanded_cv_results.query("status == 'ok'").iloc[0]

best_params_expanded = {
    "look_forward": int(best_row["look_forward"]),
    "prediction_length": int(best_row["prediction_length"]),
    "n_components": int(best_row["n_components"]),
    "covariance_type": best_row["covariance_type"],
}

print("Best expanded CV parameters:", best_params_expanded)

final_train_expanded = assemble_dataset(
    development_market,
    look_forward=best_params_expanded["look_forward"],
)

final_model_expanded = HMMDirectionClassifier(
    n_components=best_params_expanded["n_components"],
    covariance_type=best_params_expanded["covariance_type"],
    random_state=RANDOM_STATE,
).fit(final_train_expanded)

evaluation_market = pd.concat(
    [development_market, holdout_market]
).sort_index()

evaluation_frame_expanded = assemble_dataset(
    evaluation_market,
    look_forward=best_params_expanded["look_forward"],
)

holdout_predictions_expanded = rolling_predictions(
    final_model_expanded,
    evaluation_frame_expanded,
    prediction_length=best_params_expanded["prediction_length"],
    n_ahead=N_AHEAD,
    start_time=holdout_market.index.min(),
)

holdout_metrics_expanded = pd.Series(
    evaluate_predictions(holdout_predictions_expanded),
    name="expanded_features_holdout",
)

holdout_predictions_expanded.to_csv(
    "hmm_holdout_predictions_expanded_features.csv"
)

display(holdout_metrics_expanded.to_frame("value"))
display(holdout_predictions_expanded.head())

Best expanded CV parameters: {'look_forward': 40, 'prediction_length': 20, 'n_components': 3, 'covariance_type': 'diag'}


,value
regime_count,1668.000000
regime_coverage,0.976581
regime_accuracy,0.591127
regime_balanced_accuracy,0.567161
regime_f1_macro,0.534890
one_step_count,1687.000000
one_step_coverage,0.987705
one_step_accuracy,0.487848
one_step_balanced_accuracy,0.487292
one_step_f1_macro,0.431997


,prediction,label,close_t,one_step_direction,future_direction,score_long,score_short
timestamp,,,,,,,
2024-10-24 23:00:00,short,short,278350,short,short,-17.620629,-15.641773
2024-10-25 10:00:00,short,short,278125,short,short,-17.614141,-15.825241
2024-10-25 11:00:00,short,short,277975,long,short,-17.486160,-16.010349
2024-10-25 12:00:00,short,short,278725,long,short,-17.090235,-15.574643
2024-10-25 13:00:00,short,short,279825,short,short,-19.934527,-18.068863


In [8]:
BEST_PARAM_GRID = {
    "look_forward": [40],
    "prediction_length": [20],
    "n_components": [3],
    "covariance_type": ["diag"],
}

RANDOM_STATES = [7, 21, 42, 77, 101]

seed_cv_results = []

for seed in RANDOM_STATES:
    print(f"\n===== random_state={seed} =====")

    result = temporal_cv_search(
        development_market=development_market,
        param_grid=BEST_PARAM_GRID,
        n_ahead=N_AHEAD,
        n_splits=N_SPLITS,
        random_state=seed,
    )

    best_seed_result = result.query("status == 'ok'").iloc[0].copy()
    best_seed_result["random_state"] = seed
    seed_cv_results.append(best_seed_result)

seed_cv_results = pd.DataFrame(seed_cv_results)

columns_to_show = [
    "random_state",
    "valid_folds",
    "cv_regime_accuracy",
    "cv_regime_balanced_accuracy",
    "cv_regime_f1_macro",
    "cv_one_step_balanced_accuracy",
    "cv_future_balanced_accuracy",
]

display(seed_cv_results[columns_to_show])


===== random_state=7 =====
[001] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 20}; regime balanced accuracy=0.6410

===== random_state=21 =====
[001] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 20}; regime balanced accuracy=0.6375

===== random_state=42 =====
[001] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 20}; regime balanced accuracy=0.6426

===== random_state=77 =====
[001] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 20}; regime balanced accuracy=0.6395

===== random_state=101 =====
[001] ok: {'covariance_type': 'diag', 'look_forward': 40, 'n_components': 3, 'prediction_length': 20}; regime balanced accuracy=0.6381


,random_state,valid_folds,cv_regime_accuracy,cv_regime_balanced_accuracy,cv_regime_f1_macro,cv_one_step_balanced_accuracy,cv_future_balanced_accuracy
0,7,4,0.644611,0.640966,0.623971,0.495906,0.477016
0,21,4,0.640320,0.637462,0.620011,0.494750,0.474981
0,42,4,0.648337,0.642615,0.626832,0.496159,0.479669
0,77,4,0.635669,0.639518,0.617242,0.496247,0.476309
0,101,4,0.634173,0.638092,0.617741,0.497400,0.474797


In [9]:
metrics_by_seed = [
    "cv_regime_accuracy",
    "cv_regime_balanced_accuracy",
    "cv_regime_f1_macro",
    "cv_one_step_balanced_accuracy",
    "cv_future_balanced_accuracy",
]

seed_summary = seed_cv_results[metrics_by_seed].agg(
    ["mean", "std", "min", "max"]
).T

display(seed_summary)

seed_cv_results.to_csv(
    "hmm_best_configuration_random_states.csv",
    index=False,
)

,mean,std,min,max
cv_regime_accuracy,0.640622,0.005951,0.634173,0.648337
cv_regime_balanced_accuracy,0.639731,0.002105,0.637462,0.642615
cv_regime_f1_macro,0.621160,0.004136,0.617242,0.626832
cv_one_step_balanced_accuracy,0.496093,0.000945,0.494750,0.497400
cv_future_balanced_accuracy,0.476554,0.001971,0.474797,0.479669


## Interpretation

- `regime_*` compares the prediction with the retrospective extrema regime.
- `one_step_*` evaluates `close[t+1] - close[t]`.
- `future_*` evaluates `close[t+N_AHEAD] - close[t]`.
- `coverage` reports the fraction of predictions with a non-flat, available target.

Do not inspect or tune against holdout metrics repeatedly. Any subsequent model
redesign requires a new untouched holdout period or a walk-forward backtest.